In [145]:
import torch
import math
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
torch.manual_seed(0)
x=torch.rand(2000,1) * 2 * math.pi
y=torch.sin(x)
x_train, y_train= x[:1600],y[:1600]
x_test, y_test=x[1600:], y[1600:]
train_dataset=TensorDataset(x_train, y_train)
test_dataset=TensorDataset(x_test, y_test)
train_dataloader = DataLoader(train_dataset, batch_size=64)
test_dataloader = DataLoader(test_dataset, batch_size=64)



In [146]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(1*1, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork()

In [147]:
learning_rate = 0.005
batch_size = 64
epochs = 200

In [148]:
loss_fn=nn.MSELoss()

In [149]:
# Usare l'ottimizzatore Adam con un learning rate più adeguato (es. 0.005 o 0.01)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)

In [150]:
def train_loop(dataloader, model, loss_fn, optimizer):
    model.train()
    total, n = 0.0, 0
    for x, y in dataloader:
        pred = model(x)
        loss = loss_fn(pred, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total += loss.item() * len(x)
        n += len(x)
    return total / n


def test_loop(dataloader, model, loss_fn):
    model.eval()
    total, n = 0.0, 0
    with torch.no_grad():
        for x, y in dataloader:
            total += loss_fn(model(x), y).item() * len(x)
            n += len(x)
    return total / n

In [151]:
for t in range(epochs):
    train_loss = train_loop(train_dataloader, model, loss_fn, optimizer)
    if t % 20 == 0:
        test_loss = test_loop(test_dataloader, model, loss_fn)
        print(f"Epoch {t+1}: train {train_loss:.2e}  test {test_loss:.2e}")

Epoch 1: train 2.19e-01  test 1.14e-01
Epoch 21: train 1.27e-04  test 3.60e-05
Epoch 41: train 6.39e-04  test 1.76e-03
Epoch 61: train 6.41e-06  test 3.81e-06
Epoch 81: train 3.94e-06  test 4.53e-06
Epoch 101: train 1.12e-04  test 2.66e-05
Epoch 121: train 1.31e-04  test 3.08e-04
Epoch 141: train 3.13e-05  test 6.03e-05
Epoch 161: train 1.86e-04  test 3.59e-04
Epoch 181: train 1.02e-04  test 5.90e-05


In [152]:
xk = torch.tensor([[1.0]], requires_grad=True, dtype=torch.float)
model(xk)

tensor([[0.8390]], grad_fn=<AddmmBackward0>)

In [153]:
math.sin(1)

0.8414709848078965

In [154]:
xk = torch.tensor([[1.0]], requires_grad=True)
out = model(xk)
grad, = torch.autograd.grad(out, xk)
print(grad.item(), math.cos(1.0))

0.5375860929489136 0.5403023058681398


In [156]:
import time

def timed_run(device, epochs=200, seed=0):
    torch.manual_seed(seed)
    # Istanziamo la classe corretta NeuralNetwork e la spostiamo sul device
    model_local = NeuralNetwork().to(device)
    optimizer = torch.optim.Adam(model_local.parameters(), lr=0.005)
    xtr, ytr = x_train.to(device), y_train.to(device)   # dati sul device UNA volta sola
    loader = DataLoader(TensorDataset(xtr, ytr), batch_size=64, shuffle=True)

    if device == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(epochs):
        train_loop(loader, model_local, loss_fn, optimizer)
    if device == "cuda":
        torch.cuda.synchronize()
    return time.perf_counter() - t0

# Eseguiamo il benchmark se la GPU è disponibile, altrimenti usiamo la CPU
device_to_test = "cuda" if torch.cuda.is_available() else "cpu"
timed_run(device_to_test, epochs=5)                  # warm-up, da scartare
print("cpu :", timed_run("cpu"))
if torch.cuda.is_available():
    print("cuda:", timed_run("cuda"))
else:
    print("cuda: Non disponibile su questa istanza")

cpu : 8.863084203999733
cuda: 11.023960919999809


In [158]:
torch.manual_seed(0)
x10, y10 = x_train[:10], y_train[:10]
loader10 = DataLoader(TensorDataset(x10, y10), batch_size=10, shuffle=True)


model_local = NeuralNetwork()
optimizer = torch.optim.Adam(model_local.parameters(), lr=0.005)
for t in range(2000):
    train_loss = train_loop(loader10, model_local, loss_fn, optimizer)
    if t % 200 == 0 or t == 1999:
        test_loss = test_loop(test_dataloader, model_local, loss_fn)
        print(f"Epoch {t+1}: train {train_loss:.2e}  test {test_loss:.2e}")

Epoch 1: train 4.66e-01  test 5.78e-01
Epoch 201: train 3.37e-03  test 1.55e-02
Epoch 401: train 1.37e-05  test 2.48e-03
Epoch 601: train 4.48e-05  test 4.35e-03
Epoch 801: train 1.31e-07  test 2.40e-03
Epoch 1001: train 2.13e-08  test 2.34e-03
Epoch 1201: train 2.01e-08  test 2.59e-03
Epoch 1401: train 3.04e-09  test 2.57e-03
Epoch 1601: train 1.27e-04  test 3.54e-03
Epoch 1801: train 9.00e-10  test 2.83e-03
Epoch 2000: train 8.85e-10  test 2.83e-03
